<a href="https://colab.research.google.com/github/lunchtrey84/MSAI_631_REPO/blob/main/MSAI631_Recommendation_System_using_AI_HCI_Project_Trey_Williams.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 align="center">
<h1 align="center"> Recommendation System using AI/HCI Project


<h3 align="center"> Trey Williams

<h3 align="center"> University of the Cumberlands

<h3 align="center"> MSAI-631-A01: Artificial Intelligence for Human-Computer Interaction
    
<h3 align="center"> Dr. Alan Dennis

In [32]:
import pandas as pd
import numpy as np

print("MovieMatch Recommendation System")
print("Libraries loaded successfully.")

MovieMatch Recommendation System
Libraries loaded successfully.


In [33]:
from google.colab import files

print("Upload movie_statistic_dataset.csv")

uploaded = files.upload()

Upload movie_statistic_dataset.csv


Saving movie_statistic_dataset.csv to movie_statistic_dataset (1).csv


In [34]:
DATA_FILE = "movie_statistic_dataset.csv"

movies_df = pd.read_csv(DATA_FILE)

print("Dataset loaded successfully.")
print(f"Movies: {len(movies_df):,}")
print(f"Columns: {len(movies_df.columns)}")

print("\nColumn names:")
for column in movies_df.columns:
    print(f" - {column}")

print("\nFirst five records:")
display(movies_df.head())

Dataset loaded successfully.
Movies: 4,380
Columns: 14

Column names:
 - movie_title
 - production_date
 - genres
 - runtime_minutes
 - director_name
 - director_professions
 - director_birthYear
 - director_deathYear
 - movie_averageRating
 - movie_numerOfVotes
 - approval_Index
 - Production budget $
 - Domestic gross $
 - Worldwide gross $

First five records:


,movie_title,production_date,genres,runtime_minutes,director_name,director_professions,director_birthYear,director_deathYear,movie_averageRating,movie_numerOfVotes,approval_Index,Production budget $,Domestic gross $,Worldwide gross $
0,Avatar: The Way of Water,2022-12-09,"Action,Adventure,Fantasy",192.0,James Cameron,"writer,producer,director",1954,alive,7.8,277543.0,7.061101,460000000,667830256,2265935552
1,Avengers: Endgame,2019-04-23,"Action,Adventure,Drama",181.0,-,-,-,-,8.4,1143642.0,8.489533,400000000,858373000,2794731755
2,Pirates of the Caribbean: On Stranger Tides,2011-05-20,"Action,Adventure,Fantasy",137.0,Rob Marshall,"director,miscellaneous,producer",1960,alive,6.6,533763.0,6.272064,379000000,241071802,1045713802
3,Avengers: Age of Ultron,2015-04-22,"Action,Adventure,Sci-Fi",141.0,Joss Whedon,"writer,producer,director",1964,alive,7.3,870573.0,7.214013,365000000,459005868,1395316979
4,Avengers: Infinity War,2018-04-25,"Action,Adventure,Sci-Fi",149.0,-,-,-,-,8.4,1091968.0,8.460958,300000000,678815482,2048359754


In [35]:
print("Dataset Shape:")
print(movies_df.shape)

print("\nData Types:")
display(movies_df.dtypes.to_frame("Data Type"))

print("\nMissing Values:")
missing_values = (
    movies_df.isnull()
    .sum()
    .sort_values(ascending=False)
    .to_frame("Missing Values")
)

display(missing_values)

print("\nUnique Values Per Column:")
unique_values = movies_df.nunique().sort_values(ascending=False)
display(unique_values.to_frame("Unique Values"))

Dataset Shape:
(4380, 14)

Data Types:


,Data Type
movie_title,object
production_date,object
genres,object
runtime_minutes,float64
director_name,object
director_professions,object
director_birthYear,object
director_deathYear,object
movie_averageRating,float64
movie_numerOfVotes,float64



Missing Values:


,Missing Values
movie_title,0
production_date,0
genres,0
runtime_minutes,0
director_name,0
director_professions,0
director_birthYear,0
director_deathYear,0
movie_averageRating,0
movie_numerOfVotes,0



Unique Values Per Column:


,Unique Values
approval_Index,4380
movie_numerOfVotes,4315
Worldwide gross $,4276
Domestic gross $,4261
movie_title,4241
production_date,2458
director_name,1835
Production budget $,403
genres,354
director_professions,317


In [36]:
# Create a working copy of the original dataset
movies = movies_df.copy()

# Convert production date into a usable datetime value
movies["production_date"] = pd.to_datetime(
    movies["production_date"],
    errors="coerce"
)

# Extract production year
movies["year"] = movies["production_date"].dt.year

# Clean text fields used by the recommendation system
text_columns = [
    "movie_title",
    "genres",
    "director_name"
]

for column in text_columns:
    movies[column] = (
        movies[column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

print("Movie data prepared successfully.")
print(f"Movies available: {len(movies):,}")

display(
    movies[
        [
            "movie_title",
            "year",
            "genres",
            "director_name",
            "movie_averageRating"
        ]
    ].head()
)

Movie data prepared successfully.
Movies available: 4,380


,movie_title,year,genres,director_name,movie_averageRating
0,Avatar: The Way of Water,2022,"Action,Adventure,Fantasy",James Cameron,7.8
1,Avengers: Endgame,2019,"Action,Adventure,Drama",-,8.4
2,Pirates of the Caribbean: On Stranger Tides,2011,"Action,Adventure,Fantasy",Rob Marshall,6.6
3,Avengers: Age of Ultron,2015,"Action,Adventure,Sci-Fi",Joss Whedon,7.3
4,Avengers: Infinity War,2018,"Action,Adventure,Sci-Fi",-,8.4


In [37]:
# Prepare genre text
movies["genres_clean"] = (
    movies["genres"]
    .str.lower()
    .str.replace(",", " ", regex=False)
    .str.replace("|", " ", regex=False)
)

# Prepare director text
movies["director_clean"] = (
    movies["director_name"]
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Combine important content features
movies["content_features"] = (
    movies["genres_clean"] + " " +
    movies["director_clean"]
)

print("Content features created.")

display(
    movies[
        [
            "movie_title",
            "genres",
            "director_name",
            "content_features"
        ]
    ].head(10)
)

Content features created.


,movie_title,genres,director_name,content_features
0,Avatar: The Way of Water,"Action,Adventure,Fantasy",James Cameron,action adventure fantasy james_cameron
1,Avengers: Endgame,"Action,Adventure,Drama",-,action adventure drama -
2,Pirates of the Caribbean: On Stranger Tides,"Action,Adventure,Fantasy",Rob Marshall,action adventure fantasy rob_marshall
3,Avengers: Age of Ultron,"Action,Adventure,Sci-Fi",Joss Whedon,action adventure sci-fi joss_whedon
4,Avengers: Infinity War,"Action,Adventure,Sci-Fi",-,action adventure sci-fi -
5,Justice League,"Action,Adventure,Fantasy",Zack Snyder,action adventure fantasy zack_snyder
6,Spectre,"Action,Adventure,Thriller",Sam Mendes,action adventure thriller sam_mendes
7,Solo: A Star Wars Story,"Action,Adventure,Sci-Fi",Ron Howard,action adventure sci-fi ron_howard
8,John Carter,"Action,Adventure,Sci-Fi",Andrew Stanton,action adventure sci-fi andrew_stanton
9,Batman v Superman: Dawn of Justice,"Action,Adventure,Sci-Fi",Zack Snyder,action adventure sci-fi zack_snyder


In [38]:
# BUILD TF-IDF CONTENT MODEL
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    movies["content_features"]
)

print("TF-IDF model created successfully.")
print(f"Movies represented: {tfidf_matrix.shape[0]:,}")
print(f"Content features: {tfidf_matrix.shape[1]:,}")

TF-IDF model created successfully.
Movies represented: 4,380
Content features: 1,963


In [39]:
# CALCULATE MOVIE SIMILARITY

from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(
    tfidf_matrix,
    tfidf_matrix
)

print("Movie similarity matrix created successfully.")
print(f"Similarity matrix shape: {cosine_sim.shape}")

Movie similarity matrix created successfully.
Similarity matrix shape: (4380, 4380)


In [40]:
# CONTENT-BASED MOVIE RECOMMENDATION FUNCTION

def recommend_similar_movies(movie_title, n=5):
    """
    Return the Top-N movies most similar to a selected movie.

    Parameters:
        movie_title (str): Movie selected by the user.
        n (int): Number of recommendations to return.

    Returns:
        DataFrame containing ranked movie recommendations.
    """

    # Case-insensitive title search
    matches = movies[
        movies["movie_title"].str.lower() == movie_title.lower()
    ]

    if matches.empty:
        print(f'Movie "{movie_title}" was not found.')
        return None

    # Use the first match if duplicate titles exist
    movie_index = matches.index[0]

    # Get similarity scores
    similarity_scores = list(
        enumerate(cosine_sim[movie_index])
    )

    # Sort from most similar to least similar
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the selected movie itself
    similarity_scores = [
        item for item in similarity_scores
        if item[0] != movie_index
    ]

    # Select Top-N recommendations
    top_matches = similarity_scores[:n]

    recommendation_indices = [
        index for index, score in top_matches
    ]

    similarity_values = [
        score for index, score in top_matches
    ]

    # Build results table
    results = movies.loc[
        recommendation_indices,
        [
            "movie_title",
            "year",
            "genres",
            "director_name",
            "movie_averageRating",
            "approval_Index"
        ]
    ].copy()

    results["similarity_score"] = similarity_values

    results.insert(
        0,
        "rank",
        range(1, len(results) + 1)
    )

    return results.reset_index(drop=True)

In [41]:
# PREPARE NUMERIC RANKING FEATURES

from sklearn.preprocessing import MinMaxScaler

# Create log-adjusted vote count
movies["log_votes"] = np.log1p(
    movies["movie_numerOfVotes"]
)

# Numeric features used for recommendation ranking
ranking_features = [
    "movie_averageRating",
    "approval_Index",
    "log_votes"
]

scaler = MinMaxScaler()

scaled_values = scaler.fit_transform(
    movies[ranking_features]
)

movies["rating_scaled"] = scaled_values[:, 0]
movies["approval_scaled"] = scaled_values[:, 1]
movies["votes_scaled"] = scaled_values[:, 2]

print("Numeric ranking features prepared successfully.")

display(
    movies[
        [
            "movie_title",
            "movie_averageRating",
            "movie_numerOfVotes",
            "approval_Index",
            "rating_scaled",
            "approval_scaled",
            "votes_scaled"
        ]
    ].head()
)

Numeric ranking features prepared successfully.


,movie_title,movie_averageRating,movie_numerOfVotes,approval_Index,rating_scaled,approval_scaled,votes_scaled
0,Avatar: The Way of Water,7.8,277543.0,7.061101,0.807692,0.694009,0.825323
1,Avengers: Endgame,8.4,1143642.0,8.489533,0.884615,0.842734,0.934116
2,Pirates of the Caribbean: On Stranger Tides,6.6,533763.0,6.272064,0.653846,0.611856,0.875569
3,Avengers: Age of Ultron,7.3,870573.0,7.214013,0.743590,0.709929,0.913155
4,Avengers: Infinity War,8.4,1091968.0,8.460958,0.884615,0.839758,0.930564


In [42]:
# HYBRID CONTENT + QUALITY RECOMMENDATION FUNCTION


def recommend_movies(movie_title, n=5):
    """
    Generate Top-N movie recommendations using content
    similarity combined with rating, approval, and popularity.

    Parameters:
        movie_title (str): Movie selected by the user.
        n (int): Number of recommendations to return.

    Returns:
        DataFrame containing ranked recommendations.
    """

    # --------------------------------------------------------
    # Find selected movie
    # --------------------------------------------------------

    matches = movies[
        movies["movie_title"].str.lower()
        == movie_title.lower()
    ]

    if matches.empty:
        print(f'Movie "{movie_title}" was not found.')
        return None

    movie_index = matches.index[0]

    # --------------------------------------------------------
    # Get similarity scores for selected movie
    # --------------------------------------------------------

    similarity_values = cosine_sim[movie_index]

    # Create temporary recommendation table
    candidates = movies.copy()

    candidates["similarity_score"] = similarity_values

    # Remove selected movie
    candidates = candidates[
        candidates.index != movie_index
    ].copy()

    # --------------------------------------------------------
    # Calculate final recommendation score
    # --------------------------------------------------------

    candidates["recommendation_score"] = (
        0.70 * candidates["similarity_score"]
        + 0.15 * candidates["rating_scaled"]
        + 0.10 * candidates["approval_scaled"]
        + 0.05 * candidates["votes_scaled"]
    )

    # --------------------------------------------------------
    # Rank recommendations
    # --------------------------------------------------------

    candidates = candidates.sort_values(
        by="recommendation_score",
        ascending=False
    )

    results = candidates[
        [
            "movie_title",
            "year",
            "genres",
            "director_name",
            "movie_averageRating",
            "movie_numerOfVotes",
            "approval_Index",
            "similarity_score",
            "recommendation_score"
        ]
    ].head(n).copy()

    results.insert(
        0,
        "rank",
        range(1, len(results) + 1)
    )

    return results.reset_index(drop=True)

In [43]:
# CHECK DIRECTOR DATA QUALITY


director_missing = (
    movies["director_name"]
    .astype(str)
    .str.strip()
    .isin(["", "-", "nan", "None"])
)

usable_directors = (~director_missing).sum()
unusable_directors = director_missing.sum()

print(f"Total movies: {len(movies):,}")
print(f"Movies with usable director data: {usable_directors:,}")
print(f"Movies without usable director data: {unusable_directors:,}")

print(
    f"Usable director percentage: "
    f"{usable_directors / len(movies) * 100:.2f}%"
)

Total movies: 4,380
Movies with usable director data: 4,054
Movies without usable director data: 326
Usable director percentage: 92.56%


In [44]:
# NATURAL-LANGUAGE GENRE MAPPING

genre_aliases = {
    "funny": "Comedy",
    "comedy": "Comedy",
    "scary": "Horror",
    "horror": "Horror",
    "romantic": "Romance",
    "romance": "Romance",
    "action": "Action",
    "adventure": "Adventure",
    "dramatic": "Drama",
    "drama": "Drama",
    "thriller": "Thriller",
    "suspense": "Thriller",
    "mystery": "Mystery",
    "crime": "Crime",
    "fantasy": "Fantasy",
    "sci-fi": "Sci-Fi",
    "science fiction": "Sci-Fi",
    "animated": "Animation",
    "animation": "Animation",
    "family": "Family",
    "documentary": "Documentary",
    "war": "War",
    "western": "Western",
    "musical": "Musical"
}

print(f"Genre aliases loaded: {len(genre_aliases)}")

Genre aliases loaded: 24


In [45]:
# EXTRACT USER PREFERENCES FROM NATURAL LANGUAGE

import re

def extract_preferences(user_input):
    """
    Extract movie preferences from conversational user input.

    Currently recognizes:
        - Genre descriptions
        - Movies released after a year
        - Movies released before a year
        - Movies from a specific year
        - Minimum rating requests
    """

    text = user_input.lower()

    preferences = {
        "genres": [],
        "after_year": None,
        "before_year": None,
        "year": None,
        "minimum_rating": None
    }

    # --------------------------------------------------------
    # Detect genres and descriptive genre aliases
    # --------------------------------------------------------

    for phrase, genre in genre_aliases.items():
        if phrase in text:
            preferences["genres"].append(genre)

    # Remove duplicate genres while preserving order
    preferences["genres"] = list(
        dict.fromkeys(preferences["genres"])
    )

    # --------------------------------------------------------
    # Detect year preferences
    # --------------------------------------------------------

    after_match = re.search(
        r"(?:after|since)\s+(\d{4})",
        text
    )

    before_match = re.search(
        r"before\s+(\d{4})",
        text
    )

    from_match = re.search(
        r"(?:from|in)\s+(\d{4})",
        text
    )

    if after_match:
        preferences["after_year"] = int(
            after_match.group(1)
        )

    elif before_match:
        preferences["before_year"] = int(
            before_match.group(1)
        )

    elif from_match:
        preferences["year"] = int(
            from_match.group(1)
        )

    # --------------------------------------------------------
    # Detect minimum rating
    #
    # Examples:
    # "rated above 7"
    # "rating over 8"
    # "at least 7.5"
    # --------------------------------------------------------

    rating_match = re.search(
        r"(?:rated|rating)?\s*"
        r"(?:above|over|at least)\s*"
        r"(\d(?:\.\d)?)",
        text
    )

    if rating_match:
        preferences["minimum_rating"] = float(
            rating_match.group(1)
        )

    return preferences

In [46]:
# CONVERSATIONAL MOVIE RECOMMENDATION ENGINE

def recommend_from_request(user_input, n=5):
    """
    Generate ranked movie recommendations from a
    conversational natural-language request.

    Parameters:
        user_input (str): User's movie request.
        n (int): Number of recommendations to return.

    Returns:
        DataFrame containing Top-N recommendations.
    """

    # Extract preferences from conversational input
    preferences = extract_preferences(user_input)

    # Begin with all movies
    candidates = movies.copy()

    # --------------------------------------------------------
    # FILTER BY GENRE
    # --------------------------------------------------------


    for genre in preferences["genres"]:

        candidates = candidates[
            candidates["genres"].str.contains(
                genre,
                case=False,
                na=False,
                regex=False
            )
        ]

    # --------------------------------------------------------
    # FILTER BY YEAR
    # --------------------------------------------------------

    if preferences["after_year"] is not None:

        candidates = candidates[
            candidates["year"]
            > preferences["after_year"]
        ]

    if preferences["before_year"] is not None:

        candidates = candidates[
            candidates["year"]
            < preferences["before_year"]
        ]

    if preferences["year"] is not None:

        candidates = candidates[
            candidates["year"]
            == preferences["year"]
        ]

    # --------------------------------------------------------
    # FILTER BY MINIMUM RATING
    # --------------------------------------------------------

    if preferences["minimum_rating"] is not None:

        candidates = candidates[
            candidates["movie_averageRating"]
            >= preferences["minimum_rating"]
        ]

    # --------------------------------------------------------
    # HANDLE NO RESULTS
    # --------------------------------------------------------

    if candidates.empty:

        print(
            "MovieMatch could not find a movie that "
            "matches all of those preferences."
        )

        return None

    # --------------------------------------------------------
    # RANK MATCHING MOVIES
    # --------------------------------------------------------

    candidates = candidates.copy()

    candidates["recommendation_score"] = (
        0.50 * candidates["rating_scaled"]
        + 0.30 * candidates["approval_scaled"]
        + 0.20 * candidates["votes_scaled"]
    )

    candidates = candidates.sort_values(
        by="recommendation_score",
        ascending=False
    )

    # Select fields shown to the user
    results = candidates[
        [
            "movie_title",
            "year",
            "genres",
            "director_name",
            "movie_averageRating",
            "movie_numerOfVotes",
            "approval_Index",
            "recommendation_score"
        ]
    ].head(n).copy()

    results.insert(
        0,
        "rank",
        range(1, len(results) + 1)
    )

    return results.reset_index(drop=True)

In [47]:
# GENERATE HUMAN-CENTERED EXPLANATIONS

def explain_recommendation(movie, preferences):
    """
    Create a short explanation describing why a movie
    matches the user's stated preferences.
    """

    reasons = []

    # Explain genre match
    if preferences["genres"]:

        genre_text = " and ".join(
            preferences["genres"]
        )

        reasons.append(
            f"matches your interest in {genre_text}"
        )

    # Explain year match
    if preferences["after_year"] is not None:

        reasons.append(
            f"was released after "
            f"{preferences['after_year']}"
        )

    elif preferences["before_year"] is not None:

        reasons.append(
            f"was released before "
            f"{preferences['before_year']}"
        )

    elif preferences["year"] is not None:

        reasons.append(
            f"was released in "
            f"{preferences['year']}"
        )

    # Explain rating
    if preferences["minimum_rating"] is not None:

        reasons.append(
            f"has a rating of "
            f"{movie['movie_averageRating']:.1f}"
        )

    elif movie["movie_averageRating"] >= 8.0:

        reasons.append(
            f"has a strong rating of "
            f"{movie['movie_averageRating']:.1f}"
        )

    # Combine explanation
    if reasons:

        return "Recommended because it " + ", ".join(reasons) + "."

    return (
        "Recommended based on its overall rating, "
        "approval, and popularity."
    )

In [48]:

# MOVIEMATCH USER RESPONSE

def moviematch(user_input, n=5):
    """
    Main MovieMatch recommendation interface.
    """

    print("MovieMatch")
    print("-" * 60)

    print(f'You asked: "{user_input}"')

    preferences = extract_preferences(
        user_input
    )

    results = recommend_from_request(
        user_input,
        n=n
    )

    if results is None:
        return

    print(
        f"\nI found {len(results)} recommendations "
        f"for you:\n"
    )

    for _, movie in results.iterrows():

        print(
            f"{int(movie['rank'])}. "
            f"{movie['movie_title']} "
            f"({int(movie['year'])})"
        )

        print(
            f"   Genres: {movie['genres']}"
        )

        print(
            f"   Rating: "
            f"{movie['movie_averageRating']:.1f}/10"
        )

        explanation = explain_recommendation(
            movie,
            preferences
        )

        print(
            f"   Why: {explanation}"
        )

        print()

    return results

In [49]:
# USER PREFERENCE PROFILE

user_profile = {
    "liked_movies": [],
    "disliked_movies": [],
    "liked_genres": {},
    "disliked_genres": {}
}

print("MovieMatch user preference profile initialized.")

MovieMatch user preference profile initialized.


In [50]:
# RECORD USER FEEDBACK

def record_feedback(movie_title, liked=True):
    """
    Record whether the user liked or disliked a movie.

    Duplicate feedback for the same movie is ignored so that
    repeated execution does not artificially increase genre
    preference scores.
    """

    matches = movies[
        movies["movie_title"].str.lower()
        == movie_title.lower()
    ]

    if matches.empty:
        print(f'Movie "{movie_title}" was not found.')
        return

    movie = matches.iloc[0]

    # Use the dataset's exact title
    canonical_title = movie["movie_title"]

    # Check whether feedback already exists
    already_liked = (
        canonical_title in user_profile["liked_movies"]
    )

    already_disliked = (
        canonical_title in user_profile["disliked_movies"]
    )

    if already_liked or already_disliked:
        print(
            f'Feedback for "{canonical_title}" '
            f'has already been recorded.'
        )
        return

    movie_genres = [
        genre.strip()
        for genre in movie["genres"].split(",")
    ]

    if liked:

        user_profile["liked_movies"].append(
            canonical_title
        )

        for genre in movie_genres:

            user_profile["liked_genres"][genre] = (
                user_profile["liked_genres"].get(
                    genre,
                    0
                ) + 1
            )

        print(
            f'Feedback recorded: '
            f'You liked "{canonical_title}".'
        )

    else:

        user_profile["disliked_movies"].append(
            canonical_title
        )

        for genre in movie_genres:

            user_profile["disliked_genres"][genre] = (
                user_profile["disliked_genres"].get(
                    genre,
                    0
                ) + 1
            )

        print(
            f'Feedback recorded: '
            f'You disliked "{canonical_title}".'
        )

In [51]:
# PERSONALIZED RECOMMENDATIONS FROM USER FEEDBACK

def recommend_from_profile(n=5):
    """
    Generate personalized recommendations using movies and
    genres the user has liked during the current session.
    """

    if not user_profile["liked_movies"]:

        print(
            "MovieMatch needs at least one liked movie "
            "before creating personalized recommendations."
        )

        return None

    # Start with all movies
    candidates = movies.copy()

    # Do not recommend movies already rated by the user
    excluded_movies = (
        user_profile["liked_movies"]
        + user_profile["disliked_movies"]
    )

    candidates = candidates[
        ~candidates["movie_title"].isin(
            excluded_movies
        )
    ].copy()

    # --------------------------------------------------------
    # Calculate genre preference score
    # --------------------------------------------------------

    def calculate_genre_preference(genres):

        movie_genres = [
            genre.strip()
            for genre in genres.split(",")
        ]

        score = 0

        for genre in movie_genres:

            score += user_profile[
                "liked_genres"
            ].get(genre, 0)

            score -= user_profile[
                "disliked_genres"
            ].get(genre, 0)

        return score

    candidates["preference_score"] = (
        candidates["genres"].apply(
            calculate_genre_preference
        )
    )

    # Normalize preference score
    max_preference = (
        candidates["preference_score"]
        .abs()
        .max()
    )

    if max_preference > 0:

        candidates["preference_scaled"] = (
            candidates["preference_score"]
            / max_preference
        )

    else:

        candidates["preference_scaled"] = 0

    # --------------------------------------------------------
    # Personalized ranking
    # --------------------------------------------------------

    candidates["personalized_score"] = (
        0.50 * candidates["preference_scaled"]
        + 0.25 * candidates["rating_scaled"]
        + 0.15 * candidates["approval_scaled"]
        + 0.10 * candidates["votes_scaled"]
    )

    candidates = candidates.sort_values(
        "personalized_score",
        ascending=False
    )

    results = candidates[
        [
            "movie_title",
            "year",
            "genres",
            "movie_averageRating",
            "approval_Index",
            "preference_score",
            "personalized_score"
        ]
    ].head(n).copy()

    results.insert(
        0,
        "rank",
        range(1, len(results) + 1)
    )

    return results.reset_index(drop=True)

In [52]:

# ADAPTIVE CONTENT-BASED RECOMMENDATION MODEL


def recommend_from_profile(n=5):
    """
    Generate personalized recommendations using positive and
    negative feedback from the current MovieMatch session.
    """

    if not user_profile["liked_movies"]:

        print(
            "MovieMatch needs at least one liked movie "
            "before creating personalized recommendations."
        )

        return None

    candidates = movies.copy()

    # --------------------------------------------------------
    # Exclude movies already evaluated by the user
    # --------------------------------------------------------

    evaluated_movies = (
        user_profile["liked_movies"]
        + user_profile["disliked_movies"]
    )

    candidates = candidates[
        ~candidates["movie_title"].isin(
            evaluated_movies
        )
    ].copy()

    # --------------------------------------------------------
    # Find dataset indices for liked movies
    # --------------------------------------------------------

    liked_indices = movies[
        movies["movie_title"].isin(
            user_profile["liked_movies"]
        )
    ].index.tolist()

    # Average similarity to all liked movies
    liked_similarity = cosine_sim[
        :,
        liked_indices
    ].mean(axis=1)

    candidates["liked_similarity"] = (
        liked_similarity[candidates.index]
    )

    # --------------------------------------------------------
    # Find dataset indices for disliked movies
    # --------------------------------------------------------

    disliked_indices = movies[
        movies["movie_title"].isin(
            user_profile["disliked_movies"]
        )
    ].index.tolist()

    if disliked_indices:

        disliked_similarity = cosine_sim[
            :,
            disliked_indices
        ].mean(axis=1)

        candidates["disliked_similarity"] = (
            disliked_similarity[candidates.index]
        )

    else:

        candidates["disliked_similarity"] = 0.0

    # --------------------------------------------------------
    # Calculate personalized score
    # --------------------------------------------------------

    candidates["personalized_score"] = (
        0.40 * candidates["liked_similarity"]
        - 0.20 * candidates["disliked_similarity"]
        + 0.20 * candidates["rating_scaled"]
        + 0.10 * candidates["approval_scaled"]
        + 0.10 * candidates["votes_scaled"]
    )

    candidates = candidates.sort_values(
        by="personalized_score",
        ascending=False
    )

    # --------------------------------------------------------
    # Prepare results
    # --------------------------------------------------------

    results = candidates[
        [
            "movie_title",
            "year",
            "genres",
            "movie_averageRating",
            "liked_similarity",
            "disliked_similarity",
            "personalized_score"
        ]
    ].head(n).copy()

    results.insert(
        0,
        "rank",
        range(1, len(results) + 1)
    )

    return results.reset_index(drop=True)

In [53]:
# EXPLAIN PERSONALIZED RECOMMENDATIONS


def explain_personalized_recommendation(movie):
    """
    Generate a human-readable explanation for a personalized
    recommendation based on the user's current feedback.
    """

    reasons = []

    # Get movie genres
    movie_genres = {
        genre.strip()
        for genre in movie["genres"].split(",")
    }

    # --------------------------------------------------------
    # Explain similarities to liked movies
    # --------------------------------------------------------

    for liked_title in user_profile["liked_movies"]:

        liked_match = movies[
            movies["movie_title"] == liked_title
        ]

        if liked_match.empty:
            continue

        liked_movie = liked_match.iloc[0]

        liked_genres = {
            genre.strip()
            for genre in liked_movie["genres"].split(",")
        }

        shared_genres = sorted(
            movie_genres.intersection(liked_genres)
        )

        same_director = (
            movie.get("director_name", "") ==
            liked_movie["director_name"]
            and liked_movie["director_name"] not in ["", "-"]
        )

        if same_director:

            reasons.append(
                f"it has the same director as {liked_title}"
            )

        if shared_genres:

            reasons.append(
                "it shares "
                + ", ".join(shared_genres)
                + f" with {liked_title}"
            )

    # --------------------------------------------------------
    # Add rating information when particularly strong
    # --------------------------------------------------------

    if movie["movie_averageRating"] >= 8.0:

        reasons.append(
            f"it has a strong rating of "
            f"{movie['movie_averageRating']:.1f}/10"
        )

    # Keep explanations concise
    if reasons:
        return "Recommended because " + "; ".join(reasons[:3]) + "."

    return (
        "Recommended based on your previous movie feedback "
        "and the movie's overall quality."
    )

In [54]:
# DISPLAY PERSONALIZED MOVIEMATCH RESULTS

def personalized_moviematch(n=5):
    """
    Display personalized MovieMatch recommendations using
    human-readable explanations instead of model statistics.
    """

    results = recommend_from_profile(n=n)

    if results is None:
        return None

    print("MovieMatch Personalized Recommendations")
    print("-" * 60)

    for _, result in results.iterrows():

        # Retrieve complete movie record for explanation
        movie_match = movies[
            movies["movie_title"]
            == result["movie_title"]
        ]

        if movie_match.empty:
            continue

        movie = movie_match.iloc[0]

        print(
            f"{int(result['rank'])}. "
            f"{result['movie_title']} "
            f"({int(result['year'])})"
        )

        print(
            f"   Genres: {result['genres']}"
        )

        print(
            f"   Rating: "
            f"{result['movie_averageRating']:.1f}/10"
        )

        explanation = (
            explain_personalized_recommendation(movie)
        )

        print(
            f"   Why: {explanation}"
        )

        print()

    return results

In [55]:
# DISPLAY CURRENT USER PREFERENCES

def show_preferences():
    """
    Display the feedback currently stored by MovieMatch
    during this session.
    """

    print("\nMovieMatch Current Preferences")
    print("-" * 40)

    if user_profile["liked_movies"]:
        print(
            "Liked movies: "
            + ", ".join(user_profile["liked_movies"])
        )
    else:
        print("Liked movies: None")

    if user_profile["disliked_movies"]:
        print(
            "Disliked movies: "
            + ", ".join(user_profile["disliked_movies"])
        )
    else:
        print("Disliked movies: None")

In [56]:

# MOVIEMATCH CONVERSATIONAL CONTROLLER

def run_moviematch():
    """
    Run the complete MovieMatch recommendation system.
    """

    print("=" * 60)
    print("MOVIEMATCH")
    print("Human-Centered Movie Recommendation System")
    print("=" * 60)

    print(
        "\nTell MovieMatch what you want to watch, "
        "or use one of the available commands."
    )

    print(
        "\nCommands: recommend, personal, like, dislike, "
        "profile, reset, help, exit"
    )

    while True:

        command = input(
            "\nMovieMatch > "
        ).strip().lower()

        # ----------------------------------------------------
        # EXIT
        # ----------------------------------------------------

        if command in ["exit", "quit"]:

            print(
                "\nThanks for using MovieMatch. I hope you enjoy you time in the Cinema"
            )
            break

        # ----------------------------------------------------
        # HELP
        # ----------------------------------------------------

        elif command == "help":

            print("\nAvailable Commands")
            print("-" * 40)
            print(
                "recommend - Request movies using "
                "natural language"
            )
            print(
                "personal  - Recommendations based "
                "on your feedback"
            )
            print(
                "like      - Record a movie you liked"
            )
            print(
                "dislike   - Record a movie you disliked"
            )
            print(
                "profile   - View your current preferences"
            )
            print(
                "reset     - Clear your preferences"
            )
            print(
                "exit      - End MovieMatch"
            )

        # ----------------------------------------------------
        # NATURAL-LANGUAGE RECOMMENDATION
        # ----------------------------------------------------

        elif command == "recommend":

            request = input(
                "\nWhat kind of movie would you like? "
            ).strip()

            if not request:

                print(
                    "Please enter a movie preference."
                )
                continue

            moviematch(
                request,
                n=5
            )

        # ----------------------------------------------------
        # PERSONALIZED RECOMMENDATION
        # ----------------------------------------------------

        elif command == "personal":

            personalized_moviematch(
                n=5
            )

        # ----------------------------------------------------
        # POSITIVE FEEDBACK
        # ----------------------------------------------------

        elif command == "like":

            title = input(
                "\nWhat movie did you like? "
            ).strip()

            if title:
                record_feedback(
                    title,
                    liked=True
                )

        # ----------------------------------------------------
        # NEGATIVE FEEDBACK
        # ----------------------------------------------------

        elif command == "dislike":

            title = input(
                "\nWhat movie did you dislike? "
            ).strip()

            if title:
                record_feedback(
                    title,
                    liked=False
                )

        # ----------------------------------------------------
        # VIEW PROFILE
        # ----------------------------------------------------

        elif command == "profile":

            show_preferences()

        # ----------------------------------------------------
        # RESET PROFILE
        # ----------------------------------------------------

        elif command == "reset":

            reset_user_profile()

        # ----------------------------------------------------
        # ALLOW DIRECT NATURAL-LANGUAGE REQUESTS
        # ----------------------------------------------------
        #
        # If the user does not enter a recognized command,
        # MovieMatch treats the text as a movie request.
        #
        # Example:
        # "Give me an action movie after 2015"
        # ----------------------------------------------------

        else:

            moviematch(
                command,
                n=5
            )

In [57]:

# LAUNCH MOVIEMATCH

run_moviematch()

MOVIEMATCH
Human-Centered Movie Recommendation System

Tell MovieMatch what you want to watch, or use one of the available commands.

Commands: recommend, personal, like, dislike, profile, reset, help, exit

MovieMatch > exit

Thanks for using MovieMatch. I hope you enjoy you time in the Cinema


## Transistion to Graphics Interface.

In [62]:
# ============================================================
# MOVIEMATCH GUI CONVERSATIONAL FUNCTIONS
# ============================================================

import gradio as gr


def gui_help():
    """
    Display instructions for using MovieMatch.
    """

    return (
        "MOVIEMATCH HELP\n"
        "==================================================\n\n"
        "AVAILABLE COMMANDS\n\n"
        "help\n"
        "Display MovieMatch help.\n\n"

        "profile\n"
        "View your current liked and disliked movies.\n\n"

        "reset\n"
        "Clear your current MovieMatch preferences.\n\n"

        "personal\n"
        "Get recommendations based on your feedback.\n\n"

        "like <movie title>\n"
        "Tell MovieMatch that you liked a movie.\n"
        "Example: like Licorice Pizza\n\n"

        "dislike <movie title>\n"
        "Tell MovieMatch that you disliked a movie.\n"
        "Example: dislike Love Actually\n\n"

        "MOVIE REQUESTS\n\n"
        "You can also enter a normal movie request.\n\n"

        "Examples:\n"
        "Give me a romantic comedy after 2018\n"
        "Find an action movie rated above 8\n"
        "Show me a scary movie before 2000\n\n"

        "You may also use the buttons in the "
        "My Preferences tab."
    )


def gui_profile():
    """
    Display the current MovieMatch preference profile.
    """

    liked = user_profile["liked_movies"]
    disliked = user_profile["disliked_movies"]

    liked_text = ", ".join(liked) if liked else "None"
    disliked_text = ", ".join(disliked) if disliked else "None"

    return (
        "CURRENT MOVIEMATCH PREFERENCES\n"
        "==================================================\n\n"
        f"Liked Movies: {liked_text}\n\n"
        f"Disliked Movies: {disliked_text}"
    )


def gui_like(movie_title):
    """
    Record positive feedback for a movie.
    """

    if not movie_title or not movie_title.strip():
        return "Enter a movie title first."

    movie_title = movie_title.strip()

    # Check whether the movie exists.
    matches = movies[
        movies["movie_title"].str.lower()
        == movie_title.lower()
    ]

    if matches.empty:
        return f'Movie "{movie_title}" was not found.'

    canonical_title = matches.iloc[0]["movie_title"]

    # Check for existing feedback.
    if (
        canonical_title in user_profile["liked_movies"]
        or canonical_title in user_profile["disliked_movies"]
    ):
        return (
            f'Feedback for "{canonical_title}" '
            "has already been recorded.\n\n"
            + gui_profile()
        )

    # Existing MovieMatch function uses a Boolean value.
    record_feedback(
        canonical_title,
        liked=True
    )

    return (
        f'Feedback recorded: You liked "{canonical_title}".\n\n'
        + gui_profile()
    )


def gui_dislike(movie_title):
    """
    Record negative feedback for a movie.
    """

    if not movie_title or not movie_title.strip():
        return "Enter a movie title first."

    movie_title = movie_title.strip()

    # Check whether the movie exists.
    matches = movies[
        movies["movie_title"].str.lower()
        == movie_title.lower()
    ]

    if matches.empty:
        return f'Movie "{movie_title}" was not found.'

    canonical_title = matches.iloc[0]["movie_title"]

    # Check for existing feedback.
    if (
        canonical_title in user_profile["liked_movies"]
        or canonical_title in user_profile["disliked_movies"]
    ):
        return (
            f'Feedback for "{canonical_title}" '
            "has already been recorded.\n\n"
            + gui_profile()
        )

    # Existing MovieMatch function uses a Boolean value.
    record_feedback(
        canonical_title,
        liked=False
    )

    return (
        f'Feedback recorded: You disliked "{canonical_title}".\n\n'
        + gui_profile()
    )


def gui_reset():
    """
    Clear all MovieMatch feedback from the current session.
    """

    user_profile["liked_movies"].clear()
    user_profile["disliked_movies"].clear()
    user_profile["liked_genres"].clear()
    user_profile["disliked_genres"].clear()

    return (
        "Your MovieMatch preferences have been reset.\n\n"
        + gui_profile()
    )


def gui_personalized():
    """
    Generate personalized recommendations using the user's
    current MovieMatch feedback.
    """

    results = recommend_from_profile(n=5)

    if results is None:
        return (
            "MovieMatch needs at least one liked movie before "
            "creating personalized recommendations."
        )

    output = []

    for _, result in results.iterrows():

        movie_match = movies[
            movies["movie_title"]
            == result["movie_title"]
        ]

        if movie_match.empty:
            continue

        movie = movie_match.iloc[0]

        explanation = (
            explain_personalized_recommendation(movie)
        )

        output.append(
            f"{int(result['rank'])}. "
            f"{result['movie_title']} "
            f"({int(result['year'])})\n"
            f"Genres: {result['genres']}\n"
            f"Rating: "
            f"{result['movie_averageRating']:.1f}/10\n"
            f"Why: {explanation}"
        )

    return "\n\n".join(output)


def gui_recommend(user_request):
    """
    Process natural-language movie requests and MovieMatch
    commands entered in the main recommendation textbox.
    """

    if not user_request:
        return (
            "Please describe what kind of movie you want "
            "or type 'help' for instructions."
        )

    user_request = user_request.strip()

    if not user_request:
        return (
            "Please describe what kind of movie you want "
            "or type 'help' for instructions."
        )

    command = user_request.lower()

    # --------------------------------------------------------
    # HELP
    # --------------------------------------------------------

    if command == "help":
        return gui_help()

    # --------------------------------------------------------
    # PROFILE
    # --------------------------------------------------------

    if command == "profile":
        return gui_profile()

    # --------------------------------------------------------
    # RESET
    # --------------------------------------------------------

    if command == "reset":
        return gui_reset()

    # --------------------------------------------------------
    # PERSONALIZED RECOMMENDATIONS
    # --------------------------------------------------------

    if command in ["personal", "personalized"]:
        return gui_personalized()

    # --------------------------------------------------------
    # LIKE COMMAND
    # --------------------------------------------------------

    if command.startswith("like "):
        movie_title = user_request[5:].strip()
        return gui_like(movie_title)

    # --------------------------------------------------------
    # DISLIKE COMMAND
    # --------------------------------------------------------

    if command.startswith("dislike "):
        movie_title = user_request[8:].strip()
        return gui_dislike(movie_title)

    # --------------------------------------------------------
    # NORMAL NATURAL-LANGUAGE MOVIE REQUEST
    # --------------------------------------------------------

    results = recommend_from_request(
        user_request,
        n=5
    )

    if results is None or results.empty:
        return (
            "MovieMatch could not find movies matching "
            "those preferences.\n\n"
            "Try broadening your request or type 'help'."
        )

    preferences = extract_preferences(
        user_request
    )

    output = []

    for _, result in results.iterrows():

        explanation = explain_recommendation(
            result,
            preferences
        )

        output.append(
            f"{int(result['rank'])}. "
            f"{result['movie_title']} "
            f"({int(result['year'])})\n"
            f"Genres: {result['genres']}\n"
            f"Rating: "
            f"{result['movie_averageRating']:.1f}/10\n"
            f"Why: {explanation}"
        )

    return "\n\n".join(output)


print("MovieMatch GUI functions loaded successfully.")

MovieMatch GUI functions loaded successfully.


In [63]:
# ============================================================
# BUILD MOVIEMATCH GRAPHICAL USER INTERFACE
# ============================================================

with gr.Blocks(title="MovieMatch") as app:

    gr.Markdown(
        """
        # MovieMatch

        ### Human-Centered Movie Recommendation System

        Describe the type of movie you want to watch, or provide
        feedback so MovieMatch can personalize future recommendations.

        **Example:**
        *Give me a romantic comedy after 2018*

        You can also type **help** for MovieMatch commands.
        """
    )

    # ========================================================
    # FIND MOVIES TAB
    # ========================================================

    with gr.Tab("Find Movies"):

        movie_request = gr.Textbox(
            label="What would you like to watch?",
            placeholder=(
                "Example: Give me a funny action movie "
                "after 2010 or type help"
            ),
            lines=2
        )

        recommend_button = gr.Button(
            "Get Recommendations"
        )

        recommendation_output = gr.Textbox(
            label="MovieMatch Recommendations",
            lines=18,
            interactive=False
        )

        recommend_button.click(
            fn=gui_recommend,
            inputs=movie_request,
            outputs=recommendation_output
        )

        # Pressing Enter can also submit the request.
        movie_request.submit(
            fn=gui_recommend,
            inputs=movie_request,
            outputs=recommendation_output
        )

    # ========================================================
    # MY PREFERENCES TAB
    # ========================================================

    with gr.Tab("My Preferences"):

        feedback_movie = gr.Textbox(
            label="Movie Title",
            placeholder="Example: Licorice Pizza"
        )

        with gr.Row():

            like_button = gr.Button(
                "👍 Like"
            )

            dislike_button = gr.Button(
                "👎 Dislike"
            )

        profile_output = gr.Textbox(
            label="Current MovieMatch Profile",
            lines=8,
            interactive=False
        )

        like_button.click(
            fn=gui_like,
            inputs=feedback_movie,
            outputs=profile_output
        )

        dislike_button.click(
            fn=gui_dislike,
            inputs=feedback_movie,
            outputs=profile_output
        )

        # ----------------------------------------------------
        # PERSONALIZED RECOMMENDATIONS
        # ----------------------------------------------------

        gr.Markdown(
            "### Personalized Recommendations"
        )

        personalized_button = gr.Button(
            "Recommend From My Feedback"
        )

        personalized_output = gr.Textbox(
            label="Personalized Results",
            lines=18,
            interactive=False
        )

        personalized_button.click(
            fn=gui_personalized,
            inputs=[],
            outputs=personalized_output
        )

        # ----------------------------------------------------
        # PROFILE AND HELP CONTROLS
        # ----------------------------------------------------

        gr.Markdown(
            "## Preference Controls"
        )

        with gr.Row():

            profile_button = gr.Button(
                "View Profile"
            )

            reset_button = gr.Button(
                "Reset Preferences"
            )

            help_button = gr.Button(
                "Help"
            )

        control_output = gr.Textbox(
            label="MovieMatch Information",
            lines=15,
            interactive=False
        )

        profile_button.click(
            fn=gui_profile,
            inputs=[],
            outputs=control_output
        )

        reset_button.click(
            fn=gui_reset,
            inputs=[],
            outputs=control_output
        )

        help_button.click(
            fn=gui_help,
            inputs=[],
            outputs=control_output
        )

In [64]:
# ============================================================
# LAUNCH MOVIEMATCH GUI
# ============================================================

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fb57088757e995d5df.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
